# LangExtract: Extracting Structured Data from Clinical Notes

This notebook shows how to use [LangExtract](https://github.com/google/langextract), Google's open-source Python library, to extract **diagnoses, medications and dosages** from free-text clinical notes using an LLM.

Every extraction is grounded to its exact character position in the source text, and the results can be explored in an interactive HTML visualization.

## Why LangExtract instead of prompting an LLM directly?

Under the hood, LangExtract **does** prompt an LLM: it builds a prompt from your instructions and examples, sends it to the model and processes the response. The value is everything it handles around that call:

| | Direct LLM prompt | LangExtract |
|---|---|---|
| **Source grounding** | Returns values with no link to where they appear | Maps every extraction to its exact character span in the source text |
| **Hallucination detection** | Hard to tell if a value was invented | Values that can't be aligned to the text are easy to spot |
| **Long documents** | Limited by the context window; details in the middle get missed | Automatic chunking, parallel processing and multiple passes for higher recall |
| **Consistent output** | You write the JSON parsing and handle malformed responses | Schema defined by few-shot examples; output parsed into typed objects |
| **Auditability** | Manual review | Interactive HTML visualization highlighting each extraction in context |
| **Scale** | Custom loop, retries and storage | Batch processing, retries and JSONL export built in |
| **Portability** | Code tied to one provider's SDK | Swap Gemini, OpenAI or local models (Ollama) by changing `model_id` |

**Rule of thumb:** for a one-off extraction from a short text, a direct prompt (ideally with structured output) is enough. LangExtract pays off with **long documents, many documents, or when you must prove where each value came from** (clinical, legal, financial or compliance use cases).

## 1. Setup

Install the library and set your Gemini API key (get one at [Google AI Studio](https://aistudio.google.com/app/apikey)).

In [1]:
%pip install -q langextract

In [2]:
import os
import textwrap

import langextract as lx
from google.colab import userdata

os.environ["LANGEXTRACT_API_KEY"] = userdata.get("LANGEXTRACT_API_KEY")

## 2. Define the extraction task

Write a clear prompt describing **what** to extract and **how** (use exact text, don't paraphrase).

In [3]:
prompt = textwrap.dedent("""\
    Extract diagnoses, medications and dosages from the clinical note.
    Use the exact text as it appears; do not paraphrase or invent data.
    Add useful attributes (e.g. frequency or route of administration).
""")

## 3. Provide few-shot examples

One or more high-quality examples define the output schema: the extraction classes and the attributes the model should return.

In [4]:
examples = [
    lx.data.ExampleData(
        text="Patient with arterial hypertension. Start losartan 50 mg once daily.",
        extractions=[
            lx.data.Extraction(
                extraction_class="diagnosis",
                extraction_text="arterial hypertension",
                attributes={"status": "active"},
            ),
            lx.data.Extraction(
                extraction_class="medication",
                extraction_text="losartan",
                attributes={"route": "oral"},
            ),
            lx.data.Extraction(
                extraction_class="dosage",
                extraction_text="50 mg once daily",
                attributes={"medication": "losartan", "frequency": "every 24 hours"},
            ),
        ],
    )
]

## 4. Input text

In [5]:
input_text = textwrap.dedent("""\
    58-year-old woman with type 2 diabetes mellitus and dyslipidemia.
    Continue metformin 850 mg twice daily and add atorvastatin 20 mg
    at bedtime. Follow-up in 3 months.
""")
print(input_text)

58-year-old woman with type 2 diabetes mellitus and dyslipidemia.
Continue metformin 850 mg twice daily and add atorvastatin 20 mg
at bedtime. Follow-up in 3 months.



## 5. Run the extraction

For long documents you can tune `extraction_passes`, `max_workers` and `max_char_buffer` to improve recall and speed.

In [6]:
result = lx.extract(
    text_or_documents=input_text,
    prompt_description=prompt,
    examples=examples,
    model_id="gemini-3.8-flash",
    # extraction_passes=2,
    # max_workers=10,
    # max_char_buffer=1000,
)

LangExtract: model=gemini-3.8-flash, current=165 chars, processed=0 chars:  [00:07]


## 6. Inspect the results

Each extraction includes its class, the exact text, its attributes and its character span in the source.

In [7]:
for e in result.extractions:
    span = (
        f"[{e.char_interval.start_pos}:{e.char_interval.end_pos}]"
        if e.char_interval
        else "[no span]"
    )
    print(f"{e.extraction_class:<11} {span:<10} {e.extraction_text!r}  {e.attributes or ''}")

diagnosis   [23:47]    'type 2 diabetes mellitus'  {'status': 'active'}
diagnosis   [52:64]    'dyslipidemia'  {'status': 'active'}
medication  [75:84]    'metformin'  {'route': 'oral'}
dosage      [85:103]   '850 mg twice daily'  {'medication': 'metformin', 'frequency': 'twice daily'}
medication  [112:124]  'atorvastatin'  {'route': 'oral'}
dosage      [125:141]  '20 mg at bedtime'  {'medication': 'atorvastatin', 'frequency': 'at bedtime'}


Optionally, turn the results into a pandas DataFrame for further analysis:

In [8]:
import pandas as pd

df = pd.DataFrame(
    [
        {
            "class": e.extraction_class,
            "text": e.extraction_text,
            "start": e.char_interval.start_pos if e.char_interval else None,
            "end": e.char_interval.end_pos if e.char_interval else None,
            **(e.attributes or {}),
        }
        for e in result.extractions
    ]
)
df

,class,text,start,end,status,route,medication,frequency
0,diagnosis,type 2 diabetes mellitus,23,47,active,NaN,NaN,NaN
1,diagnosis,dyslipidemia,52,64,active,NaN,NaN,NaN
2,medication,metformin,75,84,NaN,oral,NaN,NaN
3,dosage,850 mg twice daily,85,103,NaN,NaN,metformin,twice daily
4,medication,atorvastatin,112,124,NaN,oral,NaN,NaN
5,dosage,20 mg at bedtime,125,141,NaN,NaN,atorvastatin,at bedtime


## 7. Save and visualize

Save the annotated document as JSONL and render the interactive HTML visualization, which highlights each extraction in the original text.

In [9]:
lx.io.save_annotated_documents([result], output_name="results.jsonl", output_dir=".")

html = lx.visualize("results.jsonl")

with open("visualization.html", "w", encoding="utf-8") as f:
    f.write(html.data if hasattr(html, "data") else html)

html  # renders inline in Jupyter / Colab

LangExtract: Saving to results.jsonl: 1 docs [00:00, 577.89 docs/s]

✓ Saved 1 documents to results.jsonl



LangExtract: Loading results.jsonl: 100%|██████████| 1.84k/1.84k [00:00<00:00, 3.32MB/s]

✓ Loaded 1 documents from results.jsonl


## Next steps

- **Other models:** change `model_id` to an OpenAI model or a local model served with Ollama.
- **Long documents:** pass a URL or a long string; LangExtract chunks it and processes the chunks in parallel.
- **Other domains:** swap the prompt and examples to extract from invoices, contracts, emails, etc.